# Edge TinyML: Recyclables Classifier\nColab-ready notebook. Student: Remmy Kipruto Tumo\n

In [ ]:
# Install and imports\n!pip install -q tensorflow==2.12.0\nimport tensorflow as tf\nimport numpy as np\nimport os\nfrom tensorflow.keras import layers, models\nprint('TF', tf.__version__)

In [ ]:
# Data loading (upload dataset to /content/data with train/val folders)\nDATA_DIR = '/content/data'\nIMG_SIZE = (160,160)\nBATCH_SIZE = 32\ntrain_ds = tf.keras.preprocessing.image_dataset_from_directory(DATA_DIR+'/train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')\nval_ds = tf.keras.preprocessing.image_dataset_from_directory(DATA_DIR+'/val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')\nclass_names = train_ds.class_names\nNUM_CLASSES = len(class_names)\nAUTOTUNE = tf.data.AUTOTUNE\ntrain_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)\nval_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)\nprint('Classes:', class_names)

In [ ]:
# Build model (MobileNetV2 backbone)\nbase_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE+(3,), include_top=False, weights='imagenet')\nbase_model.trainable = False\ninputs = tf.keras.Input(shape=IMG_SIZE+(3,))\nx = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)\nx = base_model(x, training=False)\nx = layers.GlobalAveragePooling2D()(x)\nx = layers.Dropout(0.2)(x)\noutputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)\nmodel = tf.keras.Model(inputs, outputs)\nmodel.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])\nmodel.summary()

In [ ]:
# Train (quick example)\nEPOCHS = 10\nhistory = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)

In [ ]:
# Save model and convert to TFLite with integer quantization\nmodel.save('model_recyclables.h5')\ndef representative_data_gen():\n    for images, labels in train_ds.take(100):\n        batch = tf.image.resize(images, IMG_SIZE)\n        batch = tf.cast(batch, tf.float32)\n        yield [batch.numpy()]\nconverter = tf.lite.TFLiteConverter.from_keras_model(model)\nconverter.optimizations = [tf.lite.Optimize.DEFAULT]\nconverter.representative_dataset = representative_data_gen\nconverter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]\nconverter.inference_input_type = tf.uint8\nconverter.inference_output_type = tf.uint8\ntflite_model = converter.convert()\nopen('model_recyclables_quant.tflite','wb').write(tflite_model)\nprint('Saved model_recyclables_quant.tflite')

In [ ]:
# Test TFLite in Colab (simulation)\ninterpreter = tf.lite.Interpreter(model_path='model_recyclables_quant.tflite')\ninterpreter.allocate_tensors()\ninput_details = interpreter.get_input_details()\noutput_details = interpreter.get_output_details()\nfor images, labels in val_ds.take(1):\n    imgs = tf.image.resize(images, IMG_SIZE)\n    imgs_uint8 = tf.cast(imgs, tf.uint8).numpy()\n    interpreter.set_tensor(input_details[0]['index'], imgs_uint8)\n    interpreter.invoke()\n    preds = interpreter.get_tensor(output_details[0]['index'])\n    print('Preds shape:', preds.shape)\n    break

**Notes:** For Raspberry Pi, install `tflite-runtime` or `tensorflow` and copy the `.tflite` file to the device.